# Regresi Multivariate

- Nama : ACHMAD RIDHO FA'IZ
- NIM : 230411100197
- Kelas : Machine Learning A

Studi kasus: memprediksi harga rumah (Harga) berdasarkan tiga fitur — Luas, Kamar, dan Jarak —
dengan 5 data sampel. Koefisien regresi dicari lewat pendekatan matriks
`W = (X^T X)^-1 X^T Y` dan diverifikasi terhadap model scikit-learn.

### Untuk siapa notebook ini

Notebook ini ditulis untuk mahasiswa yang baru mulai belajar machine learning.
Jangan kuatir dengan istilah matematis yang terlihat rumit — setiap langkah dijelaskan
dengan bahasa sehari-hari beserta arti angka yang muncul di output.

### Cerita di balik kasusnya

Bayangkan hendak membeli rumah. Sebelum ditanya harganya, kita mengira-ngira:
“kalau luasnya besar, kamarnya banyak, dan dekat dengan pusat kota, pasti lebih mahal.”
Regresi linear adalah cara mengubah kira-kira itu menjadi rumus matematis yang bisa
menghitung harga secara numerik.

Kita punya tiga informasi sebuah rumah (disebut **fitur**):

- `Luas` — luas bangunan (di contoh ini bernilai 1–5),
- `Kamar` — jumlah kamar tidur (2–4),
- `Jarak` — jarak ke pusat kota (1 = paling dekat, 5 = paling jauh),

dan satu informasi yang ingin kita tebak, yaitu **Harga**.

Pertanyaan yang dijawab notebook ini: *kalau tahu Luas, Kamar, dan Jarak sebuah rumah,
berapa kira-kira harganya?* Caranya, mencari “bidang” yang paling dekat dengan semua
titik data. Semakin kecil jarak antara harga asli dan harga tebakan, semakin bagus modelnya.


## 1. Import Library

**Apa itu library?** Kumpulan kode siap pakai agar kita tidak perlu menulis ulang hal-hal
umum seperti perhitungan matriks atau tabel data dari nol. Ibarat memasak, library adalah
bahan bumbu yang tinggal dicampur.

Tiga library utama di sini:

- **numpy** (`import numpy as np`) — jagoan angka dan matriks. Semua operasi perkalian
  matriks (`@`) dan fungsi `np.linalg.*` berasal dari sini.
- **pandas** (`import pandas as pd`) — jagoan tabel. Tabel (`DataFrame`) di langkah 2
  dibuat dengan pandas.
- **scikit-learn** (beberapa `from sklearn ... import ...`) — jagoan machine learning.
  Dari sini diambil `LinearRegression` (modelnya), `train_test_split` (pembagi
  data latih/uji), serta `mean_squared_error` dan `mean_absolute_error` (pengukur
  kualitas model).

Baris terakhir mencetak versi tiap library:

```
numpy       : 2.5.2
pandas      : 3.0.5
scikit-learn: 1.9.0
```

**Arti output:** ketiga library berhasil dimuat dan siap dipakai. Versi tercetak hanya
informasi identitas, supaya kalau ada masalah kita tahu persis versi yang berjalan.


In [1]:
import numpy as np
import pandas as pd
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn_version)


numpy       : 2.5.2
pandas      : 3.0.5
scikit-learn: 1.9.0


## 2. Membuat Dataset

**Apa itu dataset?** Bahan baku yang dipakai untuk belajar. Bentuknya tabel: satu **baris**
mewakili satu objek (di sini satu rumah), satu **kolom** mewakili satu jenis informasi.

Data 5 baris ditulis sebagai kamus Python, lalu diubah menjadi tabel `DataFrame`:

| Luas | Kamar | Jarak | Harga |
|------|-------|-------|-------|
| 1 | 2 | 5 | 3 |
| 2 | 3 | 4 | 5 |
| 3 | 3 | 2 | 7 |
| 4 | 4 | 3 | 8 |
| 5 | 4 | 1 | 11 |

**Arti tiap baris:** rumah pertama luas 1, kamar 2, jarak 5 (paling jauh), harga 3.
Rumah kelima luas 5, kamar 4, jarak 1 (paling dekat), harga 11.

**Arti tiap kolom:**

- `Luas`, `Kamar`, `Jarak` = informasi yang kita miliki (disebut **fitur**),
- `Harga` = yang ingin kita tebak (disebut **target**).

Angka sengaja dibuat bulat dan sederhana supaya tidak mengaburkan konsep. Prinsip yang
sama berlaku walau data berjumlah ribuan.


In [2]:
data = {
    "Luas": [1, 2, 3, 4, 5],
    "Kamar": [2, 3, 3, 4, 4],
    "Jarak": [5, 4, 2, 3, 1],
    "Harga": [3, 5, 7, 8, 11],
}
df = pd.DataFrame(data)
df


,Luas,Kamar,Jarak,Harga
0,1,2,5,3
1,2,3,4,5
2,3,3,2,7
3,4,4,3,8
4,5,4,1,11


## 3. Menentukan X dan y

Di langkah ini kita memisahkan “apa yang kita tahu” dan “apa yang ingin dicari”.

- **`X`** (huruf besar) = kumpulan **fitur**, data masukan. Diambil dari kolom
  `Luas`, `Kamar`, `Jarak` — jadinya sebuah tabel 5 baris × 3 kolom.
- **`y`** (huruf kecil) = **target**, nilai yang ingin diprediksi. Diambil dari
  kolom `Harga` — sebuah daftar 5 angka.

**Output sel ini:**

```
X : (5, 3)
y : (5,)
```

**Arti output:** `X` berbentuk 5 baris × 3 kolom (5 rumah, 3 fitur); `y` berupa
daftar 5 harga. Bentuk `(5, 3)` adalah cara Python menyebut tabel, sedangkan `(5,)`
berarti daftar sederhana.


In [3]:
X = df[["Luas", "Kamar", "Jarak"]]
y = df["Harga"]
print("X :", X.shape)
print("y :", y.shape)


X : (5, 3)
y : (5,)


## 4. Train-Test Split

**Mengapa data dipisah?** Kalau model diuji memakai data yang sama dengan data
belajarnya, nilainya akan tampak bagus palsu — seperti siswa yang hafal kunci jawaban.
Agar penilaian jujur, sebagian data disembunyikan untuk “ujian” dan tidak pernah
dilihat ketika belajar.

- **Train (latih)** = data untuk belajar; model melihat harga aslinya di sini.
- **Test (uji)** = data untuk ujian; model menebak tanpa tahu jawaban, baru nanti
  dibandingkan dengan harga asli.

`test_size=0.2` berarti 20% data disisihkan untuk ujian (pada 5 baris → 1 baris),
sisanya 4 baris untuk latihan. `random_state=0` membuat pemilihan baris selalu sama,
jadi hasilnya bisa diulang dan dibandingkan antarkali menjalankan.

**Output sel ini:**

```
X_train : (4, 3)
X_test  : (1, 3)
```

**Arti output:** bagian latih 4 baris, bagian uji 1 baris, keduanya tetap membawa
3 fitur. Dengan `random_state=0`, rumah yang terpilih sebagai uji ternyata yang
harganya **7** — penting untuk dibaca di langkah 10.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)


X_train : (4, 3)
X_test  : (1, 3)


## 5. Membuat Model

**Penting:** membuat model ≠ melatih model.

- Membuat model = menyiapkan “wadah rumus” yang masih kosong. Di sini wadahnya
  regresi linear: `Harga = b0 + b1·Luas + b2·Kamar + b3·Jarak`. Angka `b0, b1, b2, b3`
  masih kosong dan akan diisi saat training.
- Melatih model = mengisi angka-angka itu dengan nilai terbaik (dilakukan di langkah 6).

`LinearRegression()` dipanggil tanpa argumen, artinya memakai pengaturan standar.

**Output sel ini:**

```
LinearRegression()
```

**Arti output:** objek model berhasil dibuat. Tanda kurung kosong berarti tidak ada
pengaturan tambahan. Model masih “kosong” — belum mempelajari apa pun.


In [5]:
model = LinearRegression()
model


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


## 6. Training

**Apa yang terjadi saat `model.fit(...)`?** Model disodori `X_train` (fitur 4 rumah)
bersama `y_train` (harga aslinya), lalu mencari kombinasi `b0, b1, b2, b3` yang membuat
prediksi sedekat mungkin dengan harga asli. Ini setara dengan cara matematis
`W = (X^T X)^-1 X^T y` (metode least squares) yang dipakai untuk verifikasi di langkah 7.

Setelah melatih, `model.score(...)` mengukur kecocokan model terhadap data latih dalam
bentuk nilai **R² (R-squared)**: semakin dekat ke 1 (atau 100%), semakin cocok.

**Output sel ini:**

```
R2 latih : 1.0000
```

**Arti output:** nilai `1.0000` (≈100%) berarti model *sempurna* menjelaskan 4 data
latih. Wajar karena titiknya cuma 4 dan memang bisa dibuat tepat oleh satu persamaan
linear — jarang terjadi pada data dunia nyata. Angka ini hanya cek latihan; penilaian
sesungguhnya ada pada data uji (langkah 9–11).


In [6]:
model.fit(X_train, y_train)
r2_latih = model.score(X_train, y_train)
print(f"R2 latih : {r2_latih:.4f}")


R2 latih : 1.0000


## 7. Mendapatkan b0, b1, b2, b3

Setelah training, angka-angka di dalam model bisa “dibongkar”:

- `b0 = model.intercept_` → **intercept**: harga tebakan ketika semua fitur bernilai 0
  (titik awal persamaan).
- `b1, b2, b3 = model.coef_` → **koefisien**: seberapa kuat tiap fitur memengaruhi harga
  (`b1` untuk Luas, `b2` untuk Kamar, `b3` untuk Jarak).

**Mengapa dihitung ulang dengan matriks?** Untuk memverifikasi. Rumus dasar metode
least squares adalah `W = (X^T X)^-1 X^T y`, yang di sini dirakit dengan:

- `X_mat` = tabel fitur + kolom tambahan berisi angka 1 (tempat `b0`),
- `X_mat.T @ X_mat` = `X^T X` (matriks X dikali transposnya),
- `np.linalg.inv(...)` = mencari invers matriks,
- `@ X_mat.T @ y` = mengalikan hasilnya dengan `X^T y`.

Kalau dua cara menghitung menghasilkan angka sama, berarti modelnya benar.

**Output sel ini:**

```
sklearn : [ 7.  1. -0. -1.]
matriks : [ 7.  1.  0. -1.]
```

**Arti output:** baris atas dari scikit-learn, baris bawah dari matriks; keduanya
sama: `b0 = 7`, `b1 = 1`, `b2 ≈ 0`, `b3 = -1`. (`-0` dan `0` hanya beda tampilan
pembulatan, keduanya sebenarnya nol.) Artinya dua pendekatan saling cocok.

**Makna koefisiennya:** setiap tambahan 1 satuan Luas menaikkan harga 1; jumlah Kamar
hampir tak berpengaruh; setiap kenaikan 1 satuan Jarak (makin jauh) menurunkan harga 1.


In [7]:
b0 = model.intercept_
b1, b2, b3 = model.coef_

X_mat = np.hstack([np.ones((X_train.shape[0], 1)), X_train.to_numpy()])
W = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ y_train.to_numpy()

print("sklearn :", np.round(np.array([b0, b1, b2, b3]), 4))
print("matriks :", np.round(W, 4))


sklearn : [ 7.  1. -0. -1.]
matriks : [ 7.  1.  0. -1.]


## 8. Membentuk Persamaan Regresi

Persamaan regresi adalah ringkasan jadi dari model: satu rumus untuk menghitung harga
kapankapan.

```
Y = 7.00 + (1.00)X1 + (-0.00)X2 + (-1.00)X3
- X1 : Luas
- X2 : Kamar
- X3 : Jarak
```

**Arti output, baris per baris:**

- `7.00` = `b0`, harga dasar ketika semua fitur nol.
- `(1.00)X1` — setiap 1 satuan Luas, harga bertambah 1.
- `(-0.00)X2` — Kamar praktis tidak berpengaruh (angka nyaris nol).
- `(-1.00)X3` — setiap 1 satuan Jarak bertambah (makin jauh), harga berkurang 1.
  Tanda minus ini masuk akal: rumah yang jauh dari pusat kota biasanya lebih murah.

Sebagai cerita: rumah “standar” dihargai 7. Kalau luasnya besar, harga naik; kalau
jaraknya menjauh, harga turun.


In [8]:
print(f"Y = {b0:.2f} + ({b1:.2f})X1 + ({b2:.2f})X2 + ({b3:.2f})X3")
print("- X1 : Luas")
print("- X2 : Kamar")
print("- X3 : Jarak")


Y = 7.00 + (1.00)X1 + (-0.00)X2 + (-1.00)X3
- X1 : Luas
- X2 : Kamar
- X3 : Jarak


## 9. Testing / Prediksi

Inilah saat “ujian”. `X_test` berisi satu rumah yang tidak pernah dilihat model saat
training. Model menebak harganya memakai persamaan dari langkah 8. Rumah uji tersebut
memiliki Luas 3, Kamar 3, Jarak 2 (harga asli 7).

**Output sel ini:**

```
[8.]
```

**Arti output:** model menebak harga **8**. Cek manual: `7 + 1·3 + 0·3 + (-1)·2 = 8`.
Tanda kurung `[]` dipakai karena hasil prediksi bisa berisi banyak angka bila data
ujinya banyak. Sekarang tebakan ini dibandingkan dengan harga asli di langkah 10.


In [9]:
y_pred = model.predict(X_test)
print(y_pred)


[8.]


## 10. Membandingkan Aktual vs Prediksi

Aktual = harga asli (`y_test`), Prediksi = tebakan model (`y_pred`),
Selisih = Aktual − Prediksi. Tabel hasilnya:

| Aktual | Prediksi | Selisih |
|--------|----------|---------|
| 7 | 8.0 | -1.0 |

**Cara membaca:**

- Harga asli 7, tebakan model 8.
- Selisih `-1.0` berarti model menebak **terlalu tinggi** 1 satuan.
- Tanda negatif = tebakan berlebih; positif = kurang. Makin dekat ke nol, makin baik.

Selisih inilah yang dibawa ke langkah 11 untuk diubah menjadi angka-angka evaluasi.


In [10]:
perbandingan = pd.DataFrame({
    "Aktual": y_test.values,
    "Prediksi": y_pred,
    "Selisih": y_test.values - y_pred,
})
perbandingan


,Aktual,Prediksi,Selisih
0,7,8.0,-1.0


## 11. Evaluasi Model

Satu selisih saja tidak cukup menggambarkan kualitas. Empat pengukur berikut melihat
error dari sudut berbeda:

- **MSE (Mean Squared Error)** — rata-rata selisih **dikuadratkan**. Mengkuadratkan
  membuat error besar “dihukum” lebih berat; satuannya jadi kuadrat harga.
- **RMSE (Root Mean Squared Error)** — akar dari MSE, mengembalikan satuan ke satuan
  harga asli sehingga lebih mudah dibayangkan.
- **MAE (Mean Absolute Error)** — rata-rata selisih tanpa kuadrat; ukuran error “polos”.
- **MAPE (Mean Absolute Percentage Error)** — rata-rata error dalam persen terhadap
  harga asli; paling mudah dibaca orang awam.

Rumus umum: `MSE = (1/n) Σ(y_aktual − y_prediksi)²`. Di sini `n = 1` karena satu data uji.

**Output sel ini:**

```
MSE  : 1.0000
RMSE : 1.0000
MAE  : 1.0000
MAPE : 14.29%
```

**Arti output:** dengan satu data uji, model rata-rata meleset 1 dari harga asli
(MSE, RMSE, MAE kebetulan sama karena selisihnya −1). MAPE `14.29%` berarti tebakan
meleset sekitar 14% dari harga sebenarnya. Karena data ujinya cuma satu, angka ini
lebih baik dibaca sebagai ilustrasi cara menghitung daripada kesimpulan kekuatan model.


In [11]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"MAPE : {mape:.2f}%")



MSE  : 1.0000
RMSE : 1.0000
MAE  : 1.0000
MAPE : 14.29%


## 12. Verifikasi: Mengapa MSE = 3.85 Salah

**Latar belakang:** hasil hitungan manual menuliskan koefisien `W = (-2.20, 1.80, 0.90,
-0.50)` dan `MSE = 19.25/5 = 3.85`. Aritmetika `19.25/5 = 3.85` memang benar, tetapi
pertanyaannya: **apakah angka 19.25 itu benar?**

Bagian ini membuktikannya lewat kaidah baku metode kuadrat terkecil (least squares),
bukan sekadar membandingkan kode. Logikanya sederhana: kalau `Y_hat` (prediksi) yang
dipakai untuk menghitung `19.25` bukan prediksi yang sah, maka `3.85` bukanlah MSE
yang sebenarnya.

Alur pembuktiannya:

1. Hitung `Y_hat`, residual `e`, `SSR`, dan `MSE` versi manual → memunculkan 19.25 & 3.85.
2. Uji syarat matematis least squares (normal equation) → memperlihatkan `Y_hat` manual tak sah.
3. Tunjukkan solusi yang benar beserta metriknya.


In [12]:
Y = y.values
Y_klaim = np.array([0.90, 2.10, 4.90, 7.10, 9.90])
e = Y - Y_klaim

print("Y_hat manual :", Y_klaim)
print("Residual e   :", e)
print("e^2          :", (e ** 2))
print("SSR          :", (e ** 2).sum())
print("MSE          :", (e ** 2).mean())


Y_hat manual : [0.9 2.1 4.9 7.1 9.9]
Residual e   : [2.1 2.9 2.1 0.9 1.1]
e^2          : [4.41 8.41 4.41 0.81 1.21]
SSR          : 19.25
MSE          : 3.85


### Uji normal equation

**Inti syarat ini:** metode least squares mencari `Y_hat` yang “paling dekat” dengan
data. Ada satu aturan bernama **normal equation** untuk mengeceknya:

> Jumlah residual `e` (ditimbang tiap fitur) harus sama dengan 0, atau `X^T e = 0`.

Kalau syarat terpenuhi, `sum(e²)` benar-benar nilai terkecil yang mungkin (SSR minimum).
Kalau tidak, berarti prediksi itu bukan yang terbaik — errornya masih bisa dikecilkan —
sehingga `sum(e²)` dan MSE-nya bukan nilai yang benar.

Di sel ini, `Xm` adalah `X` ditambah satu kolom berisi angka 1 (tempat intercept `b0`).
Empat kolom `Xm` (kolom 1, Luas, Kamar, Jarak) dikalikan dengan residual `e` lalu
dijumlahkan. Hasil idealnya semua nol.


In [13]:
Xm = np.hstack((np.ones((Y.shape[0], 1)), X.values))

print("X^T e (harusnya semua nol):")
print(Xm.T @ e)


X^T e (harusnya semua nol):
[ 9.1 23.3 27.2 30.1]


### Kontradiksi antara W dan Y_hat manual

Selain gagal pada uji normal equation, ada kejanggalan yang lebih mudah terlihat:
angka `Y_hat` yang dipakai menghitung `MSE = 3.85` tidak cocok dengan `W` yang diklaim.

Kalau `Y_hat` benar-benar dihitung dari `W = (-2.20, 1.80, 0.90, -0.50)` untuk baris
pertama (Luas = 1, Kamar = 2, Jarak = 5):

`-2.20 + 1.80·1 + 0.90·2 + (-0.50)·5 = -1.10` — **bukan 0.90**.

Baris pertama `Y_hat` seharusnya `-1.10`, tetapi manual menulis `0.90`. Ini indikator
kuat bahwa angka-angka manual saling tidak konsisten.


In [14]:
W_klaim = np.array([-2.20, 1.80, 0.90, -0.50])
Yhat_W = Xm @ W_klaim
e_W = Y - Yhat_W

print("Y_hat(W manual) :", Yhat_W)
print("Residual e      :", e_W)
print("SSR             :", (e_W ** 2).sum())
print("MSE             :", (e_W ** 2).mean())


Y_hat(W manual) : [-1.1  2.1  4.9  7.1  9.9]
Residual e      : [4.1 2.9 2.1 0.9 1.1]
SSR             : 31.65
MSE             : 6.33


### Mendapatkan solusi yang benar

Di sini numpy diminta mencari `W` yang benar lewat `np.linalg.solve`, yaitu
menyelesaikan persamaan `(X^T X) W = X^T y`. Caranya setara dengan rumus
`W = (X^T X)^-1 X^T y`, hanya lebih stabil secara numerik.

Langkah dalam sel ini:

1. `W = np.linalg.solve(Xm.T @ Xm, Xm.T @ Y)` → koefisien regresi yang benar.
2. `Yhat = Xm @ W` → prediksi untuk kelima rumah.
3. `e = Y - Yhat` → residual (selisih nyata).
4. `Xm.T @ e` → mengecek normal equation pada hasil ini (harus nol semua).
5. `mse`, `rmse`, `mae`, `mape` → metrik evaluasi.


In [15]:
W = np.linalg.solve(Xm.T @ Xm, Xm.T @ Y)
Yhat = Xm @ W
e = Y - Yhat

print("W (least squares):", np.round(W, 4))
print("Y_hat            :", np.round(Yhat, 2))
print("X^T e            :", np.round(Xm.T @ e, 10))

mse = (e ** 2).mean()
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {np.sqrt(mse):.4f}")
print(f"MAE  : {np.abs(e).mean():.4f}")
print(f"MAPE : {np.mean(np.abs(e / Y)) * 100:.2f}%")


W (least squares): [ 3.875  1.75  -0.375 -0.375]
Y_hat            : [ 3.    4.75  7.25  8.25 10.75]
X^T e            : [-0. -0. -0. -0.]
MSE  : 0.0500
RMSE : 0.2236
MAE  : 0.2000
MAPE : 2.79%


### Kesimpulan

**Apa yang salah pada perhitungan manual?**

1. `Y_hat = (0.90, 2.10, 4.90, 7.10, 9.90)` gagal uji normal equation (`X^T e ≠ 0`),
   jadi bukan prediksi least squares. Dengan sendirinya `sum(e²) = 19.25` dan
   `MSE = 3.85` tidak valid.
2. Angka tersebut bahkan tidak konsisten dengan `W` manual-nya sendiri (`X·W = (-1.10, ...)`;
   memakai `W` itu pun menghasilkan `SSR = 31.65`, `MSE = 6.33`).

**Apa jawaban yang benar?**

- Koefisien least squares: `W = (3.875, 1.75, -0.375, -0.375)`.
- Prediksi: `Y_hat = (3.0, 4.75, 7.25, 8.25, 10.75)`.
- Uji normal equation terpenuhi (`X^T e = 0`) — ini memang yang paling dekat dengan data.
- Metrik: `MSE = 0.05`, `RMSE = 0.2236`, `MAE = 0.20`, `MAPE = 2.79%`.

**Inti pelajaran:** MSE dihitung dari prediksi model. Kalau `W`-nya salah atau `Y_hat`-nya
tidak konsisten, hasil MSE ikut salah — walaupun pembagian `19.25/5 = 3.85` terlihat benar.
Yang penting bukan aritmetikanya, melainkan dari mana angka pembilang itu berasal.
